In [ ]:
import os, random, math, hashlib
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


import random




In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATASET_ROOT = Path("/kaggle/input")

TARGET_SIZE = 299
MAX_RESIZE = 320

BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-4
NUM_WORKERS = 2  


P_REAL = 0.50
P_GAN  = 0.25
P_DIFF = 0.25


In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png"}

def rglob_images(p: Path) -> List[Path]:
    if not p.exists():
        return []
    return [x for x in p.rglob("*") if x.suffix.lower() in IMG_EXTS]

def find_input_dir(containing: str) -> Path | None:
    """
    Search under /kaggle/input for a folder whose path contains `containing`.
    Returns the first match.
    """
    containing = containing.lower()
    for root, dirs, files in os.walk(DATASET_ROOT):
        rp = Path(root)
        if containing in str(rp).lower():
            return rp
    return None

def must_find_dir(containing: str) -> Path:
    p = find_input_dir(containing)
    if p is None:
        raise FileNotFoundError(f"Could not find any folder under {DATASET_ROOT} containing: {containing}")
    return p


In [ ]:
def find_subdir_by_suffix(suffix: str) -> Path:
    suffix = suffix.replace("\\", "/")
    for p in DATASET_ROOT.rglob("*"):
        if p.is_dir() and str(p).replace("\\", "/").endswith(suffix):
            return p
    raise FileNotFoundError(f"Could not find directory ending with: {suffix}")

def build_index():
    stylegan_real_dir = find_subdir_by_suffix("train/real")
    stylegan_fake_dir = find_subdir_by_suffix("train/fake")

    stylegan_real = rglob_images(stylegan_real_dir)
    stylegan_fake = rglob_images(stylegan_fake_dir)

    rvf_real_dir = find_subdir_by_suffix("Real")
    rvf_fake_dir = find_subdir_by_suffix("Fake")

    rvf_real = rglob_images(rvf_real_dir)
    rvf_fake = rglob_images(rvf_fake_dir)

    synth_root = must_find_dir("syntheticeye-diffusion-faces")
    synth_imgs = rglob_images(synth_root)

    sd_root = find_input_dir("stable-diffusion-face-dataset")
    sd_imgs = rglob_images(sd_root) if sd_root else []

    index = {
        "real": stylegan_real + rvf_real,
        "gan": stylegan_fake,
        "diff": rvf_fake + synth_imgs,
        "cross_diff": sd_imgs
    }

    print("\n✅ DATASET INDEX SUMMARY")
    print(f"  REAL        : {len(index['real'])}")
    print(f"  GAN         : {len(index['gan'])}")
    print(f"  DIFFUSION   : {len(index['diff'])}")
    print(f"  CROSS-GEN   : {len(index['cross_diff'])}")

    return index


index = build_index()


In [ ]:
TARGET_SIZE = 299
MAX_RESIZE = 320

def preprocess_pil(img: Image.Image) -> Image.Image:
    img = img.convert("RGB")
    w, h = img.size
    scale = MAX_RESIZE / max(w, h)
    img = img.resize((int(w*scale), int(h*scale)), Image.BICUBIC)

    w, h = img.size
    left = (w - TARGET_SIZE) // 2
    top  = (h - TARGET_SIZE) // 2
    img = img.crop((left, top, left + TARGET_SIZE, top + TARGET_SIZE))
    return img


In [ ]:
def build_binary_splits(index):
    splits = {"train": [], "val": [], "test_id": []}

    for group in ["real", "gan", "diff"]:
        label = 0 if group == "real" else 1
        paths = index[group][:]

        random.shuffle(paths)
        n = len(paths)
        t = int(0.7 * n)
        v = int(0.1 * n)

        splits["train"]   += [(p, label) for p in paths[:t]]
        splits["val"]     += [(p, label) for p in paths[t:t+v]]
        splits["test_id"] += [(p, label) for p in paths[t+v:]]

    crossgen = [(p, 1) for p in index["cross_diff"]]

    print("✅ BINARY SPLITS")
    print("Train:", len(splits["train"]))
    print("Val  :", len(splits["val"]))
    print("Test :", len(splits["test_id"]))
    print("Cross:", len(crossgen))

    return splits, crossgen

binary_splits, crossgen = build_binary_splits(index)


In [ ]:
def fft_transform(img):
    """
    img: Tensor (3,H,W) in [0,1]
    returns: (3,H,W) FFT magnitude
    """
    gray = img.mean(dim=0, keepdim=True)
    fft = torch.fft.fft2(gray)
    fft = torch.fft.fftshift(fft)
    mag = torch.log1p(torch.abs(fft))
    mag = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)
    return mag.repeat(3, 1, 1)


In [ ]:
class FFTBinaryDataset(Dataset):
    """
    label:
      0 = REAL
      1 = FAKE (GAN + DIFF)
    """
    def __init__(self, samples, size=299):
        self.samples = samples
        self.size = size

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = Image.open(path).convert("RGB")
        img = img.resize((320, 320), Image.BICUBIC)

        left = (320 - self.size) // 2
        img = img.crop((left, left, left+self.size, left+self.size))

        img = torch.from_numpy(np.array(img)).permute(2,0,1).float() / 255.0
        img = fft_transform(img)

        return img, torch.tensor(label, dtype=torch.long)


In [ ]:
class BalancedBinarySampler(Sampler):
    def __init__(self, samples):
        self.real = [i for i,(p,l) in enumerate(samples) if l == 0]
        self.fake = [i for i,(p,l) in enumerate(samples) if l == 1]
        self.n = min(len(self.real), len(self.fake)) * 2

    def __iter__(self):
        real = random.sample(self.real, self.n//2)
        fake = random.sample(self.fake, self.n//2)
        idxs = real + fake
        random.shuffle(idxs)
        return iter(idxs)

    def __len__(self):
        return self.n


In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_ds = FFTBinaryDataset(binary_splits["train"])
val_ds   = FFTBinaryDataset(binary_splits["val"])
test_ds  = FFTBinaryDataset(binary_splits["test_id"])
cross_ds = FFTBinaryDataset(crossgen)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=BalancedBinarySampler(binary_splits["train"]),
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
cross_loader = DataLoader(cross_ds, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)


In [ ]:
def train_one_epoch(epoch):
    model.train()
    total_loss = 0

    for x,y in tqdm(train_loader, desc=f"Train {epoch}"):
        x,y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch} | Train loss: {total_loss/len(train_loader):.4f}")





In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

@torch.no_grad()
def evaluate(loader, name):
    model.eval()
    ys, ps = [], []

    for x, y in tqdm(loader, desc=name):
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)              
        preds = logits.argmax(dim=1)   

        ys.extend(y.cpu().numpy())
        ps.extend(preds.cpu().numpy())

    acc  = accuracy_score(ys, ps)
    prec = precision_score(ys, ps)
    rec  = recall_score(ys, ps)
    f1   = f1_score(ys, ps)

    print(f"{name} | acc={acc:.4f} prec={prec:.4f} rec={rec:.4f} f1={f1:.4f}")


In [ ]:
EPOCHS = 4

for epoch in range(1, EPOCHS+1):
    train_one_epoch(epoch)
    evaluate(val_loader, "VAL")

print("\nFINAL EVALUATION")
evaluate(test_loader, "TEST_ID")
evaluate(cross_loader, "TEST_CROSSGEN")


In [40]:
print("\nFINAL EVALUATION")
evaluate(test_loader, "TEST_ID")
evaluate(cross_loader, "TEST_CROSSGEN")



FINAL EVALUATION


TEST_ID: 100%|██████████| 2354/2354 [14:14<00:00,  2.75it/s]


TEST_ID | acc=0.7155 prec=0.8706 rec=0.6621 f1=0.7521


TEST_CROSSGEN: 100%|██████████| 282/282 [04:51<00:00,  1.04s/it]

TEST_CROSSGEN | acc=0.9941 prec=1.0000 rec=0.9941 f1=0.9970


In [41]:
import torch
import numpy as np
from PIL import Image

def predict_single_image(
    image_path: str,
    model,
    device=DEVICE,
    threshold=0.5,
):
    """
    Binary prediction:
      0 = REAL
      1 = FAKE

    threshold applies to FAKE probability
    """

    model.eval()


    img = Image.open(image_path).convert("RGB")


    img = img.resize((320, 320), Image.BICUBIC)


    left = (320 - TARGET_SIZE) // 2
    img = img.crop((left, left, left + TARGET_SIZE, left + TARGET_SIZE))


    img = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0


    img = fft_transform(img)


    img = img.unsqueeze(0).to(device)


    with torch.no_grad():
        logits = model(img)
        probs = torch.softmax(logits, dim=1)

    p_real = probs[0, 0].item()
    p_fake = probs[0, 1].item()

    pred = 1 if p_fake >= threshold else 0
    label = "FAKE" if pred == 1 else "REAL"

    return {
        "prediction": label,
        "p_real": round(p_real, 4),
        "p_fake": round(p_fake, 4),
        "threshold": threshold
    }


In [42]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gemini-image-1.png",
    model
)

print(result)


{'prediction': 'FAKE', 'p_real': 0.0019, 'p_fake': 0.9981, 'threshold': 0.5}


In [43]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gemini-image-2.png",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0016, 'p_fake': 0.9984, 'threshold': 0.5}


In [44]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gemini-image-3.png",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0162, 'p_fake': 0.9838, 'threshold': 0.5}


In [45]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gemini-image-4.png",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0268, 'p_fake': 0.9732, 'threshold': 0.5}


In [46]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gemini-image-5.png",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0328, 'p_fake': 0.9672, 'threshold': 0.5}


In [47]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gemini-image-6.jpeg",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0015, 'p_fake': 0.9985, 'threshold': 0.5}


In [48]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gemini-image-7.jpeg",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0059, 'p_fake': 0.9941, 'threshold': 0.5}


In [49]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gemini-image-8.jpeg",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0, 'p_fake': 1.0, 'threshold': 0.5}


In [50]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gpt-image-1.png",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0012, 'p_fake': 0.9988, 'threshold': 0.5}


In [51]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gpt-image-2.png",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.002, 'p_fake': 0.998, 'threshold': 0.5}


In [52]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gpt-image-3.png",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0073, 'p_fake': 0.9927, 'threshold': 0.5}


In [53]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gpt-image-4.png",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0041, 'p_fake': 0.9959, 'threshold': 0.5}


In [54]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gpt-image-5.png",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0015, 'p_fake': 0.9985, 'threshold': 0.5}


In [55]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gpt-image-6.png",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0001, 'p_fake': 0.9999, 'threshold': 0.5}


In [56]:
result = predict_single_image(
    "/kaggle/input/fft-test-images/gpt-image-7.png",
    model
)

print(result)

{'prediction': 'FAKE', 'p_real': 0.0003, 'p_fake': 0.9997, 'threshold': 0.5}


In [57]:
MODEL_PATH = "/kaggle/working/fft_binary_resnet18.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "architecture": "resnet18",
    "input": "FFT magnitude, 299x299",
    "classes": {0: "REAL", 1: "FAKE"},
}, MODEL_PATH)

print("✅ Model saved to:", MODEL_PATH)


✅ Model saved to: /kaggle/working/fft_binary_resnet18.pth


In [58]:
import os

print("File exists:", os.path.exists("/kaggle/working/fft_binary_resnet18.pth"))
print("File size (MB):", os.path.getsize("/kaggle/working/fft_binary_resnet18.pth") / 1e6)


File exists: True
File size (MB): 44.791115


In [59]:
import shutil
import os

MODEL = "/kaggle/working/fft_binary_resnet18.pth"
ZIP_PATH = "/kaggle/working/fft_binary_resnet18_model"

print("Model exists:", os.path.exists(MODEL))

shutil.make_archive(ZIP_PATH, "zip", root_dir="/kaggle/working", base_dir="fft_binary_resnet18.pth")

print("✅ ZIP created:", ZIP_PATH + ".zip")
print("ZIP size (MB):", os.path.getsize(ZIP_PATH + ".zip") / 1e6)


Model exists: True
✅ ZIP created: /kaggle/working/fft_binary_resnet18_model.zip
ZIP size (MB): 41.500763


In [60]:
from IPython.display import FileLink

FileLink("/kaggle/working/fft_binary_resnet18_model.zip")


/kaggle/working/fft_binary_resnet18_model.zip